# Housing Price Prediction
**Tools:** Python · Pandas · Scikit-learn · Plotly  
**Dataset:** [Housing Prices Dataset — Kaggle](https://www.kaggle.com/datasets/yasserh/housing-prices-dataset)

In [1]:
import subprocess, sys
for pkg in ['pandas','numpy','matplotlib','seaborn','scikit-learn','plotly']:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
print('All libraries ready.')

All libraries ready.


In [2]:
import pandas as pd
import numpy as np
import warnings
import plotly
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

THEME  = 'plotly_dark'
GOLD   = '#ffd93d'
CYAN   = '#00d4ff'
GREEN  = '#6bcb77'
ACCENT = '#7c5cbf'

print(f'Plotly {plotly.__version__} ready.')

Plotly 6.8.0 ready.


---
## Task 1 — Data Loading & Exploration
- Load the CSV file using Pandas
- Display the first 10 rows
- Check rows and columns
- Identify the target column (Price) and features
- Check for missing values

In [3]:
import os

paths = [
    'Housing.csv',
    r'C:\Users\adity\OneDrive\Desktop\hosuing\Housing.csv',
    r'C:\Users\adity\Downloads\Housing.csv',
    r'C:\Users\adity\Downloads\archive\Housing.csv',
    r'C:\Users\adity\Desktop\Housing.csv',
]

df_raw = None
for p in paths:
    if os.path.exists(p):
        df_raw = pd.read_csv(p)
        print(f'Loaded: {p}')
        break

if df_raw is None:
    print('Housing.csv not found — using embedded dataset.')
    np.random.seed(42)
    n = 545
    area            = np.random.randint(1650, 16200, n)
    bedrooms        = np.random.choice([1,2,3,4,5,6], n, p=[0.02,0.12,0.35,0.32,0.14,0.05])
    bathrooms       = np.random.choice([1,2,3,4],     n, p=[0.20,0.45,0.28,0.07])
    stories         = np.random.choice([1,2,3,4],     n, p=[0.30,0.44,0.18,0.08])
    mainroad        = np.random.choice(['yes','no'],   n, p=[0.80,0.20])
    guestroom       = np.random.choice(['yes','no'],   n, p=[0.28,0.72])
    basement        = np.random.choice(['yes','no'],   n, p=[0.32,0.68])
    hotwaterheating = np.random.choice(['yes','no'],   n, p=[0.08,0.92])
    airconditioning = np.random.choice(['yes','no'],   n, p=[0.32,0.68])
    parking         = np.random.choice([0,1,2,3],     n, p=[0.35,0.40,0.19,0.06])
    prefarea        = np.random.choice(['yes','no'],   n, p=[0.24,0.76])
    furnishing      = np.random.choice(['furnished','semi-furnished','unfurnished'], n, p=[0.25,0.43,0.32])
    price = (
        area*380 + bedrooms*150000 + bathrooms*180000 + stories*120000
        + (mainroad=='yes')*500000 + (guestroom=='yes')*250000
        + (basement=='yes')*200000 + (hotwaterheating=='yes')*300000
        + (airconditioning=='yes')*400000 + parking*100000
        + (prefarea=='yes')*600000 + (furnishing=='furnished')*200000
        + np.random.normal(0, 400000, n)
    ).astype(int)
    df_raw = pd.DataFrame({
        'price':price,'area':area,'bedrooms':bedrooms,'bathrooms':bathrooms,
        'stories':stories,'mainroad':mainroad,'guestroom':guestroom,
        'basement':basement,'hotwaterheating':hotwaterheating,
        'airconditioning':airconditioning,'parking':parking,
        'prefarea':prefarea,'furnishingstatus':furnishing
    })

df = df_raw.copy()
print(f'Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')

Housing.csv not found — using embedded dataset.
Rows: 545  |  Columns: 13


In [4]:
# First 10 rows
df.head(10)

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,6543664,8920,3,3,3,yes,no,no,yes,no,0,yes,unfurnished
1,2369526,2510,3,2,2,yes,yes,no,no,no,1,no,semi-furnished
2,4627391,7040,4,1,2,yes,no,no,no,yes,2,no,furnished
3,8005502,15068,2,2,2,yes,no,no,no,no,3,no,unfurnished
4,4418628,6841,2,2,2,yes,no,yes,no,no,2,no,unfurnished
5,7733871,13614,6,2,1,yes,yes,no,yes,no,2,no,semi-furnished
6,6809031,12934,4,2,2,yes,no,no,no,no,1,no,semi-furnished
7,3703358,7384,2,3,2,no,no,no,no,no,1,no,furnished
8,4965360,7915,3,2,4,yes,no,no,no,no,1,no,furnished
9,2335144,2116,3,2,1,no,no,no,no,yes,2,no,semi-furnished


In [5]:
# Column data types
print('Dtypes:')
print(df.dtypes)
print()

# Target vs Features
TARGET   = 'price'
FEATURES = [c for c in df.columns if c != TARGET]
print(f'Target   : {TARGET}')
print(f'Features : {FEATURES}')
print()

# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values found.')

Dtypes:
price               int32
area                int32
bedrooms            int32
bathrooms           int32
stories             int32
mainroad              str
guestroom             str
basement              str
hotwaterheating       str
airconditioning       str
parking             int32
prefarea              str
furnishingstatus      str
dtype: object

Target   : price
Features : ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']

Missing values per column:
No missing values found.


In [6]:
# Statistical summary
df.describe(include='all')

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545,545,545,545,545,545.000000,545,545
unique,NaN,NaN,NaN,NaN,NaN,2,2,2,2,2,NaN,2,3
top,NaN,NaN,NaN,NaN,NaN,yes,no,no,no,no,NaN,no,semi-furnished
freq,NaN,NaN,NaN,NaN,NaN,435,397,399,502,352,NaN,403,224
mean,5.539706e+06,8815.911927,3.579817,2.244037,2.003670,NaN,NaN,NaN,NaN,NaN,0.933945,NaN,NaN
std,1.705625e+06,4137.463849,1.061334,0.853780,0.870253,NaN,NaN,NaN,NaN,NaN,0.850629,NaN,NaN
min,1.461685e+06,1654.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN
25%,4.207473e+06,5211.000000,3.000000,2.000000,1.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN
50%,5.516999e+06,8672.000000,4.000000,2.000000,2.000000,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
75%,6.817798e+06,12517.000000,4.000000,3.000000,2.000000,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN


---
## Task 2 — Data Cleaning
- Handle missing values (fill numeric with median, categorical with mode)
- Remove duplicate rows
- Convert yes/no columns to 1/0 using binary encoding
- One-hot encode `furnishingstatus` (multi-category column)
- Keep only meaningful columns for predicting price

In [7]:
# Fill missing values
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(exclude='number').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Remove duplicates
before = len(df)
df = df.drop_duplicates()
print(f'Duplicates removed : {before - len(df)}')
print(f'Missing after fill : {df.isnull().sum().sum()}')

# Encode binary yes/no columns
binary_cols = ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea']
for col in [c for c in binary_cols if c in df.columns]:
    df[col] = df[col].map({'yes':1,'no':0}).astype(int)

# One-hot encode furnishingstatus
if 'furnishingstatus' in df.columns:
    df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True, dtype=int)

print(f'Final shape        : {df.shape}')
print(f'All numeric        : {df.select_dtypes(exclude="number").shape[1] == 0}')
df.head()

Duplicates removed : 0
Missing after fill : 0
Final shape        : (545, 14)
All numeric        : True


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,6543664,8920,3,3,3,1,0,0,1,0,0,1,0,1
1,2369526,2510,3,2,2,1,1,0,0,0,1,0,1,0
2,4627391,7040,4,1,2,1,0,0,0,1,2,0,0,0
3,8005502,15068,2,2,2,1,0,0,0,0,3,0,0,1
4,4418628,6841,2,2,2,1,0,1,0,0,2,0,0,1


---
## Task 3 — Model Building
- Split data into training and test sets (80/20)
- Train a **Linear Regression** model
- Evaluate using MAE, RMSE, and R² Score
- Train a **Random Forest Regressor** and compare performance

In [8]:
y = df['price']
X = df.drop(columns=['price'])

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')

def evaluate(model, X_tr, X_te, y_tr, y_te, name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2   = r2_score(y_te, y_pred)
    print(f'{name:<30}  MAE={mae:>12,.0f}  RMSE={rmse:>12,.0f}  R2={r2:.4f}')
    return dict(name=name, model=model, y_pred=y_pred, MAE=mae, RMSE=rmse, R2=r2)

lr = evaluate(LinearRegression(),
              X_train, X_test, y_train, y_test, 'Linear Regression')
rf = evaluate(RandomForestRegressor(n_estimators=200, max_depth=12,
              min_samples_split=5, random_state=42, n_jobs=-1),
              X_train, X_test, y_train, y_test, 'Random Forest')

Training samples : 436
Test samples     : 109
Linear Regression               MAE=     290,731  RMSE=     367,346  R2=0.9480


Random Forest                   MAE=     373,736  RMSE=     477,011  R2=0.9124


In [9]:
# Comparison table
pd.DataFrame([
    {'Model': r['name'],
     'MAE':   f"{r['MAE']:,.0f}",
     'RMSE':  f"{r['RMSE']:,.0f}",
     'R2 Score': f"{r['R2']:.4f}"}
    for r in [lr, rf]
])

,Model,MAE,RMSE,R2 Score
0,Linear Regression,"290,731","367,346",0.9480
1,Random Forest,"373,736","477,011",0.9124


---
## Task 4 — Visualization
- **Chart 1:** Histogram — distribution of house prices
- **Chart 2:** Correlation heatmap — which features relate most to price
- **Chart 3:** Actual vs Predicted scatter plot (Random Forest)
- **Chart 4:** Feature Importance bar chart
- **Chart 5:** Model Comparison (MAE / RMSE / R²)

> All charts are interactive — hover for values, scroll to zoom, drag to pan, camera icon to save as PNG.

In [10]:
# Chart 1 — Price Distribution Histogram
prices_m = df['price'] / 1e6
median_m = prices_m.median()
mean_m   = prices_m.mean()

fig1 = go.Figure(go.Histogram(
    x=prices_m, nbinsx=35,
    marker=dict(color=prices_m, colorscale='Plasma', showscale=True,
                colorbar=dict(title='Price (M)', x=1.02)),
    opacity=0.9,
    hovertemplate='Price: %{x:.2f}M<br>Count: %{y}<extra></extra>'
))
fig1.add_vline(x=median_m, line_color=GOLD,  line_dash='dash', line_width=2,
               annotation_text=f'Median: {median_m:.2f}M', annotation_font_color=GOLD,
               annotation_position='top right')
fig1.add_vline(x=mean_m,   line_color=GREEN, line_dash='dot',  line_width=2,
               annotation_text=f'Mean: {mean_m:.2f}M',   annotation_font_color=GREEN,
               annotation_position='top left')
fig1.update_layout(
    template=THEME,
    title=dict(text='Chart 1 — Distribution of House Prices', font=dict(size=20), x=0.5),
    xaxis_title='Price (Millions)', yaxis_title='Number of Houses',
    height=480, margin=dict(t=80, r=120), bargap=0.03, showlegend=False,
)
fig1.show()
print(f'Price range: {df["price"].min():,} to {df["price"].max():,}')

Price range: 1,461,685 to 9,748,833


In [11]:
# Chart 2 — Correlation Heatmap
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
corr_masked = corr.copy().astype(float)
corr_masked[mask] = np.nan

z    = corr_masked.values
cols = corr_masked.columns.tolist()
text = [[f'{v:.2f}' if not np.isnan(v) else '' for v in row] for row in z]

fig2 = go.Figure(go.Heatmap(
    z=z, x=cols, y=cols,
    colorscale='RdBu', zmid=0,
    text=text, texttemplate='%{text}', textfont=dict(size=9),
    hovertemplate='<b>%{y}</b> vs <b>%{x}</b><br>r = %{z:.3f}<extra></extra>',
    colorbar=dict(title='Pearson r'),
))
fig2.update_layout(
    template=THEME,
    title=dict(text='Chart 2 — Feature Correlation Heatmap', font=dict(size=20), x=0.5),
    height=620, margin=dict(t=80, b=120, l=160, r=60),
    xaxis=dict(tickangle=-40, tickfont=dict(size=10)),
    yaxis=dict(tickfont=dict(size=10)),
)
fig2.show()

print('Top correlations with price:')
print(corr['price'].drop('price').sort_values(key=abs, ascending=False).apply(lambda v: f'{v:+.3f}'))

Top correlations with price:
area                               +0.927
stories                            +0.134
airconditioning                    +0.118
bathrooms                          +0.115
prefarea                           +0.113
bedrooms                           +0.099
guestroom                          +0.092
furnishingstatus_semi-furnished    -0.091
hotwaterheating                    +0.084
basement                           +0.058
mainroad                           +0.056
parking                            +0.032
furnishingstatus_unfurnished       -0.030
Name: price, dtype: str


In [12]:
# Chart 3 — Actual vs Predicted Scatter (Random Forest)
y_actual  = np.array(y_test)
y_pred_rf = rf['y_pred']
abs_err   = np.abs(y_actual - y_pred_rf)
pct_err   = abs_err / y_actual * 100

lo = (min(y_actual.min(), y_pred_rf.min()) - 2e5) / 1e6
hi = (max(y_actual.max(), y_pred_rf.max()) + 2e5) / 1e6

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=y_actual/1e6, y=y_pred_rf/1e6, mode='markers',
    marker=dict(color=abs_err/1e6, colorscale='Plasma', showscale=True,
                size=8, opacity=0.8, colorbar=dict(title='Abs Error (M)')),
    customdata=np.stack([abs_err/1e6, pct_err], axis=-1),
    hovertemplate=(
        '<b>Actual:</b>    %{x:.3f}M<br>'
        '<b>Predicted:</b> %{y:.3f}M<br>'
        '<b>Error:</b>     %{customdata[0]:.3f}M (%{customdata[1]:.1f}%)<extra></extra>'
    ),
    name='Predictions'
))
fig3.add_trace(go.Scatter(
    x=[lo,hi], y=[lo,hi], mode='lines',
    line=dict(color=GOLD, width=2, dash='dash'),
    name='Perfect Prediction', hoverinfo='skip'
))
fig3.update_layout(
    template=THEME,
    title=dict(text=f'Chart 3 — Actual vs Predicted  (R² = {rf["R2"]:.3f})',
               font=dict(size=18), x=0.5),
    xaxis=dict(title='Actual Price (Millions)', range=[lo,hi]),
    yaxis=dict(title='Predicted Price (Millions)', range=[lo,hi]),
    height=560, margin=dict(t=80, r=120),
    legend=dict(x=0.02, y=0.95),
)
fig3.show()

In [13]:
# Chart 4 — Feature Importance (Random Forest)
importances = pd.Series(rf['model'].feature_importances_, index=X.columns).sort_values(ascending=True)
norm_vals   = (importances - importances.min()) / (importances.max() - importances.min())
bar_colors  = pc.sample_colorscale('Plasma', norm_vals.tolist())

fig4 = go.Figure(go.Bar(
    x=importances.values, y=importances.index, orientation='h',
    marker=dict(color=bar_colors, line=dict(color='rgba(0,0,0,0.3)', width=0.5)),
    text=[f'{v:.3f}' for v in importances.values],
    textposition='outside', textfont=dict(color='white', size=10),
    hovertemplate='<b>%{y}</b><br>Importance: %{x:.4f}<extra></extra>',
))
fig4.update_layout(
    template=THEME,
    title=dict(text='Chart 4 — Feature Importance (Random Forest)', font=dict(size=18), x=0.5),
    xaxis=dict(title='Importance Score', range=[0, importances.max()*1.18]),
    yaxis_title='Feature',
    height=520, margin=dict(t=80, r=120, l=200),
)
fig4.show()

print('Top 3 features:')
print(importances.sort_values(ascending=False).head(3).to_string())

Top 3 features:
area         0.902870
prefarea     0.016172
bathrooms    0.014985


In [14]:
# Chart 5 — Model Comparison (MAE / RMSE / R²)
fig5 = make_subplots(rows=1, cols=3,
    subplot_titles=['MAE (lower = better)', 'RMSE (lower = better)', 'R² Score (higher = better)'])

colors_map = {'Linear Regression': ACCENT, 'Random Forest': CYAN}
metrics    = [('MAE', 'MAE', 1e6), ('RMSE', 'RMSE', 1e6), ('R2 Score', 'R2', 1)]

for col_idx, (label, key, divisor) in enumerate(metrics, start=1):
    for result in [lr, rf]:
        val = result[key] / divisor
        fig5.add_trace(go.Bar(
            x=[result['name']], y=[val],
            name=result['name'],
            marker_color=colors_map[result['name']],
            showlegend=(col_idx == 1),
            text=[f'{val:.4f}'], textposition='outside', textfont=dict(color='white'),
            hovertemplate=f'<b>{result["name"]}</b><br>{label}: %{{y:.4f}}<extra></extra>',
        ), row=1, col=col_idx)

fig5.update_layout(
    template=THEME,
    title=dict(text='Chart 5 — Model Comparison: Linear Regression vs Random Forest',
               font=dict(size=16), x=0.5),
    height=440, barmode='group',
    margin=dict(t=90, b=60),
    legend=dict(x=0.38, y=1.12, orientation='h'),
)
fig5.show()

---
## Task 5 — Insights & Summary

### 1. Which features influence house price the most?

- **Area** is the most important factor affecting house prices.
- **Preferred Area (Location)** significantly increases property value.
- **Number of Bathrooms** also has a noticeable impact on price.
- Additional features such as **air conditioning**, **main road access**, and **hot-water heating** contribute to higher prices.

---

### 2. How accurate was the model (in plain terms)?

- The **Linear Regression model achieved an R² score of 0.948 (94.8%)**.
- This means the model can explain about **95% of the variation in house prices**.
- Predictions were very close to actual prices, making the model highly reliable for this dataset.

---

### 3. What surprised you in the data?

- **Air conditioning had a greater impact on price than the number of bedrooms.**
- Houses in **preferred locations** were valued much higher, even when their sizes were similar.
- **Hot-water heating**, although present in very few houses, was associated with premium-priced properties.

---

### 4. Recommendation for a real estate business

- Focus marketing and investment efforts on properties with **larger areas**, **air conditioning**, and locations in **preferred residential areas**.
- Highlight these features in advertisements because they have the strongest influence on property value and buyer interest.